## Map  of ocean TF original grid, regridded, bias corrected
For protocol writeup: three-panel map figure as Donald designed.

11 May 2026 | EHU
- Updated 22 Jul 2026: correct map projection, add effective depth plot

In [ ]:
# imports
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import netCDF4 as nc
from netCDF4 import Dataset
import cartopy.crs as ccrs ## map projections
from mpl_toolkits.axes_grid1 import make_axes_locatable ## control colorbars

In [ ]:
# inputs
cmip_model = 'CESM2-WACCM' # cmip model
scenario = 'ssp585' # scenario

# directories
hist_dir = '/Users/eultee/Desktop/' + scenario  # historical files for example plots
plot_dir = '/Users/eultee/Desktop/' + cmip_model + '/plots/' # directory to save plots

### Make some map plots to show steps

These all have different data/grid structure, so need custom read-in for each.

Made a few examples, 3 or 4 panels, modify below as desired.

In [ ]:
# map plots of TF raw and regridded
fns = ['/TF_Omon_CESM2-WACCM_historical_r1i1p1f1_gn_185001-201412_cropped.nc',
       '/TF_Omon_CESM2-WACCM_historical_r1i1p1f1_gn_185001-201412_cropped_regrid.nc',
       '/tfQDM_additive-AllLevels-CommonGrid-CESM2-WACCM-1850_2014-IncludingPressure-20250926.nc',
       '/TF_aQDM-ISMIP_Grid-CESM2-WACCM-1850_2100-PFromStep1-20250924.nc'
      ]
       

min_TF = 0 ## lower limit for colorbars
max_TF = 10 ## upper limit for colorbars
## min 0 to show low values in the north


# --- plotting ---
### Limits of Greenland domain ###
limN           = 86.0 ## degrees N latitude
limS           = 57.0 ## degrees N latitude
limE           = 4.0 ## degrees E latitude
limW           = 274.0 ## degrees E latitude

GrIS_polar_stereo = ccrs.Stereographic(
            central_latitude=90.0,
            central_longitude=-45.0,
            false_easting=0.0,
            false_northing=0.0,
            true_scale_latitude=70.0,
            globe=ccrs.Globe('WGS84')
        )

fig, axes = plt.subplots(1, 3, layout='constrained',
                         figsize=(10, 5),
                         subplot_kw={'projection': GrIS_polar_stereo, 
                                     # 'extent':  [-65, -20, limS, limN]
                                    }
                        )

## first one: raw CESM2-WACCM
raw_file = hist_dir + fns[0]
dsraw = xr.open_dataset(raw_file)

x = dsraw['x'].values
y = dsraw['y'].values
TF = dsraw['TF'].values

sc = axes[0].scatter(x,y,c=TF[0,0,:],
                     vmin=min_TF, vmax=max_TF,
                     s=10,edgecolors='w',
                    transform=ccrs.epsg(3413))
axes[0].set_title('(a) Raw TF on model grid')


## second one: regridded
gridfile = hist_dir + fns[1]
dsgrid = xr.open_dataset(gridfile)
xgrid = dsgrid['x'].values
ygrid = dsgrid['y'].values
TFgrid = dsgrid['TF'].values

pcm_g = axes[1].pcolormesh(xgrid,ygrid,TFgrid[0,0,:,:],
                           vmin=min_TF, vmax=max_TF,
                           edgecolors='w',linewidth=0.5,
                          transform=ccrs.epsg(3413))
axes[1].set_title('(b) Raw TF on regular grid')

                  
## third one: QDM bias-corrected
qdmfile = hist_dir + fns[2]
dsqdm = xr.open_dataset(qdmfile)
xqdm = dsqdm['x'].values
yqdm = dsqdm['y'].values
TFqdm = dsqdm['TF'].values

pcm_q = axes[2].pcolormesh(xqdm,yqdm,TFqdm[0,:,:,0],
                           vmin=min_TF, vmax=max_TF,
                           edgecolors='w',linewidth=0.5,
                          transform=ccrs.epsg(3413)) ## different dim order
axes[2].set_title('(c) Bias-corrected TF on regular grid')

                  
# ## fourth one: common ISMIP grid
# ismipfile = hist_dir + fns[3]
# dsismip = xr.open_dataset(ismipfile)
# xismip = dsismip['x'].values
# yismip = dsismip['y'].values
# TFismip = dsismip['TF'].values

# pcm_i = axes[3].pcolormesh(xismip,yismip,TFismip[0,:,:],
#                            vmin=min_TF, vmax=max_TF,
#                            # edgecolors='w',linewidth=0.5 ## no white edges on a very dense grid!
#                            transform=ccrs.epsg(3413)
#                           )

# axes[3].set_title('ISMIP grid, eff. depth'.format(cmip_model))
cbar = fig.colorbar(pcm_q, ax=axes[2])
cbar.set_label('surface thermal forcing at first time slice (°C)')

# fig.suptitle(cmip_model+'_'+scenario, fontsize=16)
# plt.tight_layout()
plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_protocol-TF-biascorrect-3panel.png', dpi=300, bbox_inches="tight")
# plt.show()

## Plot effective depth
This field is called effective depth, but I am pretty sure this is actually the deepest level with defined TF for offshore points.  The effective depth field Donald plotted also includes z_eff inside the convex hull of Greenland, which this data does not.

In [ ]:
## load effective geometry (spatial look-up table)

# files
xy_eff_file = '/Users/eultee/Documents/GitHub/gris-iceocean-process/oceanTF/XY_eff.nc'
z_eff_file = '/Users/eultee/Documents/GitHub/gris-iceocean-process/oceanTF/z_eff.nc'

# effective geometry
# NB replace masked values with NaNs
X_eff = nc.Dataset(xy_eff_file).variables['X_eff'][:].filled(np.nan)
Y_eff = nc.Dataset(xy_eff_file).variables['Y_eff'][:].filled(np.nan)
z_eff = nc.Dataset(z_eff_file).variables['z_eff'][:].filled(np.nan)

# ismip coordinates at which effective geometry applies
x = nc.Dataset(xy_eff_file).variables['x'][:].filled(np.nan)
y = nc.Dataset(xy_eff_file).variables['y'][:].filled(np.nan)
X, Y = np.meshgrid(x, y)

# vertical grid of effective depths
z = np.flipud(np.unique(z_eff[z_eff<=0]))


In [ ]:
fig, ax = plt.subplots()
pcm = ax.pcolor(X_eff, Y_eff, z_eff)
plt.colorbar(pcm, label='effective depth read in (m)')
ax.set(aspect=1)

In [ ]:
## plot deepest defined CMIP depth together with extrapolated TF

fig, axes = plt.subplots(1, 3, layout='constrained',
                         figsize=(12, 5),
                         subplot_kw={'projection': GrIS_polar_stereo, 
                                     # 'extent':  [-65, -20, limS, limN]
                                    }
                        )


## effective depth read in (m)
pcm = axes[0].pcolor(
    # X_eff[::4,::4], Y_eff[::4,::4], z_eff[::4,::4],
    X_eff, Y_eff, z_eff,
               cmap='GnBu_r', vmin=-1500,
                     transform=ccrs.epsg(3413))
axes[0].set_title('(a) CMIP deepest defined TF')

cbar1 = fig.colorbar(pcm, ax=axes[0], shrink=0.75)
cbar1.set_label('[m]')


## bias corrected TF at 300m (representative shelf depth)
pcm_q = axes[1].pcolormesh(xqdm,yqdm,TFqdm[6,:,:,0], ## 7th slice (index 6) is depth 300m, based on dsqdm rep
                           vmin=min_TF, vmax=max_TF,
                           edgecolors='w',linewidth=0.5,
                          transform=ccrs.epsg(3413)) ## different dim order
axes[1].set_title('(b) Bias-corrected TF at 300 m')


## effective depth TF on common ISMIP grid
## read these in if they haven't been read in above
ismipfile = hist_dir + fns[3]
dsismip = xr.open_dataset(ismipfile)
xismip = dsismip['x'].values
yismip = dsismip['y'].values
TFismip = dsismip['TF'].values

pcm_i = axes[2].pcolormesh(xismip,yismip,TFismip[0,:,:],
                           vmin=min_TF, vmax=max_TF,
                           # edgecolors='w',linewidth=0.5 ## no white edges on a very dense grid!
                           transform=ccrs.epsg(3413)
                          )

axes[2].set_title('(c) Bias-corrected, extrapolated TF')
cbar2 = fig.colorbar(pcm_i, ax=axes[2], shrink=0.75)
cbar2.set_label('thermal forcing at first time slice [°C]')

plt.savefig(plot_dir+'plots_'+cmip_model+'_'+scenario+'_protocol-TF-extrapolate.png', dpi=300, bbox_inches="tight")